In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__results__.html
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__output__.json
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/cleaned_HVA_dataset.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/custom.css
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__results__.html
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/__output__.json
/kaggle/input/notebooks/aabdollahii/12

In [2]:
df = pd.read_csv("/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv")

In [3]:
df.head(5)

,year,filename,word_count,content,source,label,n_words
0,2003,HAM2-811011-027.ham,136,آغاز عملیات اجرایی سد جدید بر روی رودخانه کارو...,gpt,1,96
1,2003,HAM2-811011-027.ham,136,عملیات اجرایی بدنه و سرریز سد کارون ۴ که بلندت...,grok,1,92
2,2003,HAM2-811011-027.ham,136,آغاز ساخت بدنه و سرریز بلندترین سد کشور عملیات...,human,0,137
3,2003,HAM2-811011-027.ham,136,آغاز فازهای کلیدی ساخت بزرگ‌ترین سازه آبی کشور...,qwen,1,209
4,2003,HAM2-811014-088.ham,53,گزارش تازه آب ذخیره‌شده در سدهای تهران نشان می...,gpt,1,92


In [4]:
# =========================================
# 1. Imports
# =========================================

import os
import gc
import json
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


In [5]:
# =========================================
# 2. Config
# =========================================

class CFG:
    # Paths
    DATA_PATH = "/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv"
    FABERT_BASE_MODEL = "sbunlp/fabert"
    FABERT_KG_MODEL_PATH = "/kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm"
    
    OUTPUT_DIR = "/kaggle/working/outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Columns
    TEXT_COL = "content"
    LABEL_COL = "label"
    GROUP_COL = "filename"
    SOURCE_COL = "source"
    
    # Training
    MAX_LENGTH = 256
    BATCH_SIZE = 16
    EPOCHS = 3
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    GRAD_CLIP = 1.0
    
    # Split
    TEST_SIZE = 0.15
    VAL_SIZE = 0.15   # from remaining train portion
    RANDOM_STATE = 42
    
    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Mixed precision
    FP16 = torch.cuda.is_available()
    
    # Save names
    BASE_RUN_NAME = "fabert_base_hva"
    KG_RUN_NAME = "fabert_kg_hva"
    
print("Device:", CFG.DEVICE)


Device: cuda


In [6]:
# =========================================
# 3. Seed
# =========================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.RANDOM_STATE)


In [7]:
# =========================================
# 4. Load Data
# =========================================

df = pd.read_csv(CFG.DATA_PATH)

print("Shape:", df.shape)
display(df.head())

required_cols = [CFG.TEXT_COL, CFG.LABEL_COL, CFG.GROUP_COL]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Required column '{col}' not found in dataset.")

# basic cleaning
df = df.dropna(subset=[CFG.TEXT_COL, CFG.LABEL_COL, CFG.GROUP_COL]).copy()
df[CFG.TEXT_COL] = df[CFG.TEXT_COL].astype(str).str.strip()
df = df[df[CFG.TEXT_COL] != ""].copy()

# force integer labels
df[CFG.LABEL_COL] = df[CFG.LABEL_COL].astype(int)

print("After cleaning:", df.shape)
print(df[CFG.LABEL_COL].value_counts(dropna=False))


Shape: (5647, 7)


,year,filename,word_count,content,source,label,n_words
0,2003,HAM2-811011-027.ham,136,آغاز عملیات اجرایی سد جدید بر روی رودخانه کارو...,gpt,1,96
1,2003,HAM2-811011-027.ham,136,عملیات اجرایی بدنه و سرریز سد کارون ۴ که بلندت...,grok,1,92
2,2003,HAM2-811011-027.ham,136,آغاز ساخت بدنه و سرریز بلندترین سد کشور عملیات...,human,0,137
3,2003,HAM2-811011-027.ham,136,آغاز فازهای کلیدی ساخت بزرگ‌ترین سازه آبی کشور...,qwen,1,209
4,2003,HAM2-811014-088.ham,53,گزارش تازه آب ذخیره‌شده در سدهای تهران نشان می...,gpt,1,92


After cleaning: (5647, 7)
label
1    4128
0    1519
Name: count, dtype: int64


In [8]:
# =========================================
# 5. Grouped Split
# =========================================

def grouped_train_val_test_split(
    data,
    group_col,
    label_col,
    test_size=0.15,
    val_size=0.15,
    random_state=42
):
    data = data.copy().reset_index(drop=True)
    
    # First split: train_val vs test
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(gss_test.split(data, groups=data[group_col]))
    
    train_val_df = data.iloc[train_val_idx].reset_index(drop=True)
    test_df = data.iloc[test_idx].reset_index(drop=True)
    
    # Second split: train vs val
    val_relative_size = val_size / (1.0 - test_size)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_relative_size, random_state=random_state)
    train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df[group_col]))
    
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
    
    return train_df, val_df, test_df

train_df, val_df, test_df = grouped_train_val_test_split(
    df,
    group_col=CFG.GROUP_COL,
    label_col=CFG.LABEL_COL,
    test_size=CFG.TEST_SIZE,
    val_size=CFG.VAL_SIZE,
    random_state=CFG.RANDOM_STATE
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nLabel distribution:")
print("Train:\n", train_df[CFG.LABEL_COL].value_counts(normalize=True))
print("Val:\n", val_df[CFG.LABEL_COL].value_counts(normalize=True))
print("Test:\n", test_df[CFG.LABEL_COL].value_counts(normalize=True))

# leakage check
train_groups = set(train_df[CFG.GROUP_COL].unique())
val_groups = set(val_df[CFG.GROUP_COL].unique())
test_groups = set(test_df[CFG.GROUP_COL].unique())

print("\nLeakage checks:")
print("Train ∩ Val :", len(train_groups & val_groups))
print("Train ∩ Test:", len(train_groups & test_groups))
print("Val ∩ Test  :", len(val_groups & test_groups))


Train: (3936, 7)
Val  : (856, 7)
Test : (855, 7)

Label distribution:
Train:
 label
1    0.729929
0    0.270071
Name: proportion, dtype: float64
Val:
 label
1    0.733645
0    0.266355
Name: proportion, dtype: float64
Test:
 label
1    0.733333
0    0.266667
Name: proportion, dtype: float64

Leakage checks:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [9]:
# =========================================
# 6. Save Splits
# =========================================

train_df.to_csv(f"{CFG.OUTPUT_DIR}/train_split.csv", index=False)
val_df.to_csv(f"{CFG.OUTPUT_DIR}/val_split.csv", index=False)
test_df.to_csv(f"{CFG.OUTPUT_DIR}/test_split.csv", index=False)

print("Split files saved.")


Split files saved.


In [10]:
# =========================================
# 7. Class Weights
# =========================================

def compute_class_weights(labels):
    labels = np.array(labels)
    class_counts = np.bincount(labels)
    num_classes = len(class_counts)
    total = len(labels)
    
    weights = total / (num_classes * class_counts)
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_class_weights(train_df[CFG.LABEL_COL].values)
class_weights = class_weights.to(CFG.DEVICE)

print("Class weights:", class_weights)


Class weights: tensor([1.8514, 0.6850], device='cuda:0')


In [11]:
# =========================================
# 8. Dataset
# =========================================

class PersianTextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_col, label_col, max_length):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.label_col = label_col
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_col])
        label = int(row[self.label_col])
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


In [12]:
# =========================================
# 9. DataLoader Builder
# =========================================

def build_loaders(tokenizer):
    train_dataset = PersianTextDataset(
        train_df, tokenizer, CFG.TEXT_COL, CFG.LABEL_COL, CFG.MAX_LENGTH
    )
    val_dataset = PersianTextDataset(
        val_df, tokenizer, CFG.TEXT_COL, CFG.LABEL_COL, CFG.MAX_LENGTH
    )
    test_dataset = PersianTextDataset(
        test_df, tokenizer, CFG.TEXT_COL, CFG.LABEL_COL, CFG.MAX_LENGTH
    )
    
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader


In [13]:
# =========================================
# 10. Metrics
# =========================================

def compute_metrics_from_logits(labels, logits):
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    
    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    precision_bin, recall_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except:
        auc = np.nan
    
    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_machine": precision_bin,
        "recall_machine": recall_bin,
        "f1_machine": f1_bin,
        "auc": auc,
        "preds": preds,
        "probs": probs[:, 1]
    }


In [14]:
# =========================================
# 11. Training / Evaluation
# =========================================

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, device):
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(loader, desc="Train", leave=False)
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=CFG.FP16):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            logits = outputs.logits
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(loader)


@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_logits = []
    
    pbar = tqdm(loader, desc="Eval", leave=False)
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]
        
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        all_labels.extend(labels.cpu().numpy().tolist())
        all_logits.extend(logits.cpu().numpy().tolist())
    
    metrics = compute_metrics_from_logits(np.array(all_labels), np.array(all_logits))
    metrics["loss"] = total_loss / len(loader)
    
    return metrics


In [15]:
# =========================================
# 12. Model Runner
# =========================================

def run_experiment(model_name_or_path, run_name, output_dir, num_labels=2):
    print("=" * 80)
    print(f"Running: {run_name}")
    print("=" * 80)
    
    model_output_dir = os.path.join(output_dir, run_name)
    os.makedirs(model_output_dir, exist_ok=True)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels
    )
    model.to(CFG.DEVICE)
    
    train_loader, val_loader, test_loader = build_loaders(tokenizer)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY
    )
    
    total_training_steps = len(train_loader) * CFG.EPOCHS
    warmup_steps = int(CFG.WARMUP_RATIO * total_training_steps)
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )
    
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.FP16)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    history = []
    best_f1 = -1
    best_model_path = os.path.join(model_output_dir, "best_model")
    
    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch + 1}/{CFG.EPOCHS}")
        
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler, criterion, CFG.DEVICE
        )
        val_metrics = eval_model(model, val_loader, criterion, CFG.DEVICE)
        
        epoch_log = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_weighted": val_metrics["f1_weighted"],
            "val_f1_machine": val_metrics["f1_machine"],
            "val_auc": val_metrics["auc"]
        }
        history.append(epoch_log)
        
        print(json.dumps(epoch_log, indent=2, ensure_ascii=False))
        
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            print(f"Best model saved to: {best_model_path}")
    
    # load best model
    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
    best_model.to(CFG.DEVICE)
    
    test_metrics = eval_model(best_model, test_loader, criterion, CFG.DEVICE)
    
    # reports
    test_preds = test_metrics["preds"]
    test_probs = test_metrics["probs"]
    y_true = test_df[CFG.LABEL_COL].values
    
    cls_report = classification_report(y_true, test_preds, digits=4, zero_division=0, output_dict=True)
    cls_report_text = classification_report(y_true, test_preds, digits=4, zero_division=0)
    cm = confusion_matrix(y_true, test_preds)
    
    print("\nTest Classification Report:")
    print(cls_report_text)
    
    print("\nConfusion Matrix:")
    print(cm)
    
    # save metrics
    result = {
        "run_name": run_name,
        "model_path": model_name_or_path,
        "best_val_f1_macro": best_f1,
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_macro": test_metrics["precision_macro"],
        "test_recall_macro": test_metrics["recall_macro"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        "test_precision_machine": test_metrics["precision_machine"],
        "test_recall_machine": test_metrics["recall_machine"],
        "test_f1_machine": test_metrics["f1_machine"],
        "test_auc": test_metrics["auc"],
        "confusion_matrix": cm.tolist()
    }
    
    with open(os.path.join(model_output_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    
    with open(os.path.join(model_output_dir, "test_results.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    with open(os.path.join(model_output_dir, "classification_report.json"), "w", encoding="utf-8") as f:
        json.dump(cls_report, f, ensure_ascii=False, indent=2)
    
    pred_df = test_df.copy()
    pred_df["pred_label"] = test_preds
    pred_df["pred_prob_machine"] = test_probs
    pred_df.to_csv(os.path.join(model_output_dir, "test_predictions.csv"), index=False)
    
    # cleanup
    del model
    del best_model
    del tokenizer
    del train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    return result, history


In [16]:
# =========================================
# 13. Run FaBERT Base
# =========================================

base_result, base_history = run_experiment(
    model_name_or_path=CFG.FABERT_BASE_MODEL,
    run_name=CFG.BASE_RUN_NAME,
    output_dir=CFG.OUTPUT_DIR
)


Running: fabert_base_hva


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /kaggle/working/outputs/fabert_base_hva/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Eval:   0%|          | 0/54 [00:00<?, ?it/s]


Test Classification Report:
              precision    recall  f1-score   support

           0     0.9652    0.9737    0.9694       228
           1     0.9904    0.9872    0.9888       627

    accuracy                         0.9836       855
   macro avg     0.9778    0.9805    0.9791       855
weighted avg     0.9837    0.9836    0.9836       855


Confusion Matrix:
[[222   6]
 [  8 619]]


In [17]:
# =========================================
# 14. Run FaBERT-KG
# =========================================

kg_result, kg_history = run_experiment(
    model_name_or_path=CFG.FABERT_KG_MODEL_PATH,
    run_name=CFG.KG_RUN_NAME,
    output_dir=CFG.OUTPUT_DIR
)


Running: fabert_kg_hva


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/aabdollahii/12-better-fabert-finetuned/fabert_kg_mlm
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 


Epoch 1/3


Train:   0%|          | 0/246 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /kaggle/working/outputs/fabert_kg_hva/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Eval:   0%|          | 0/54 [00:00<?, ?it/s]


Test Classification Report:
              precision    recall  f1-score   support

           0     0.9735    0.9649    0.9692       228
           1     0.9873    0.9904    0.9889       627

    accuracy                         0.9836       855
   macro avg     0.9804    0.9777    0.9790       855
weighted avg     0.9836    0.9836    0.9836       855


Confusion Matrix:
[[220   8]
 [  6 621]]


In [18]:
# =========================================
# 15. Final Comparison
# =========================================

comparison_df = pd.DataFrame([base_result, kg_result])

comparison_cols = [
    "run_name",
    "best_val_f1_macro",
    "test_accuracy",
    "test_f1_macro",
    "test_f1_weighted",
    "test_precision_machine",
    "test_recall_machine",
    "test_f1_machine",
    "test_auc"
]

comparison_df = comparison_df[comparison_cols]
comparison_df.to_csv(f"{CFG.OUTPUT_DIR}/final_comparison.csv", index=False)

display(comparison_df)


,run_name,best_val_f1_macro,test_accuracy,test_f1_macro,test_f1_weighted,test_precision_machine,test_recall_machine,test_f1_machine,test_auc
0,fabert_base_hva,0.982015,0.983626,0.979125,0.983648,0.990400,0.987241,0.988818,0.996429
1,fabert_kg_hva,0.991007,0.983626,0.979008,0.983603,0.987281,0.990431,0.988854,0.996359


In [19]:
# =========================================
# 16. Per-source evaluation
# =========================================

def per_source_report(pred_csv_path, source_col=CFG.SOURCE_COL, label_col=CFG.LABEL_COL):
    pred_df = pd.read_csv(pred_csv_path)
    
    reports = []
    for src in sorted(pred_df[source_col].dropna().unique()):
        sub = pred_df[pred_df[source_col] == src].copy()
        if len(sub) == 0:
            continue
        
        y_true = sub[label_col].values
        y_pred = sub["pred_label"].values
        
        acc = accuracy_score(y_true, y_pred)
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", pos_label=1, zero_division=0
        )
        mf1 = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )[2]
        
        reports.append({
            "source": src,
            "n_samples": len(sub),
            "accuracy": acc,
            "precision_machine": p,
            "recall_machine": r,
            "f1_machine": f1,
            "f1_macro": mf1
        })
    
    return pd.DataFrame(reports)

base_source_df = per_source_report(
    f"{CFG.OUTPUT_DIR}/{CFG.BASE_RUN_NAME}/test_predictions.csv"
)
kg_source_df = per_source_report(
    f"{CFG.OUTPUT_DIR}/{CFG.KG_RUN_NAME}/test_predictions.csv"
)

base_source_df.to_csv(f"{CFG.OUTPUT_DIR}/base_per_source.csv", index=False)
kg_source_df.to_csv(f"{CFG.OUTPUT_DIR}/kg_per_source.csv", index=False)

print("FaBERT Base per-source:")
display(base_source_df)

print("FaBERT-KG per-source:")
display(kg_source_df)


FaBERT Base per-source:


,source,n_samples,accuracy,precision_machine,recall_machine,f1_machine,f1_macro
0,gpt,171,0.982456,1.0,0.982456,0.991150,0.495575
1,grok,228,0.986842,1.0,0.986842,0.993377,0.496689
2,human,228,0.973684,0.0,0.000000,0.000000,0.493333
3,qwen,228,0.991228,1.0,0.991228,0.995595,0.497797


FaBERT-KG per-source:


,source,n_samples,accuracy,precision_machine,recall_machine,f1_machine,f1_macro
0,gpt,171,0.994152,1.0,0.994152,0.997067,0.498534
1,grok,228,0.986842,1.0,0.986842,0.993377,0.496689
2,human,228,0.964912,0.0,0.000000,0.000000,0.491071
3,qwen,228,0.991228,1.0,0.991228,0.995595,0.497797


In [20]:
# =========================================
# 17. Save Summary
# =========================================

summary = {
    "config": {
        "data_path": CFG.DATA_PATH,
        "fabert_base_model": CFG.FABERT_BASE_MODEL,
        "fabert_kg_model_path": CFG.FABERT_KG_MODEL_PATH,
        "max_length": CFG.MAX_LENGTH,
        "batch_size": CFG.BATCH_SIZE,
        "epochs": CFG.EPOCHS,
        "learning_rate": CFG.LEARNING_RATE,
        "weight_decay": CFG.WEIGHT_DECAY,
        "warmup_ratio": CFG.WARMUP_RATIO,
        "random_state": CFG.RANDOM_STATE
    },
    "results": {
        "fabert_base": base_result,
        "fabert_kg": kg_result
    }
}

with open(f"{CFG.OUTPUT_DIR}/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("All outputs saved to:", CFG.OUTPUT_DIR)


All outputs saved to: /kaggle/working/outputs
